# Group Activity Recognition - Baseline 4
This notebook implements **Baseline 4 (B4: Temporal Model with Person Features)**.

### Architecture Details:
1. **Player Tracking**: Pretrained Faster R-CNN detector + IoU association tracker (SORT/ByteTrack equivalent) to track the 12 players forward and backward across 9 frames starting from the ground-truth middle frame bounding boxes.
2. **Feature Extraction**: Player crops are resized to `(224, 224)` and passed through a ResNet-50 backbone to get 2048-dimensional features.
3. **Temporal Player Modeling**: An LSTM layer processes the sequence of 9 frames at the player crop level.
4. **Group Representation**: Temporal player features are pooled (Max-pooling) over all individuals.
5. **Group Activity Classifier**: Linear head maps group features to the 8 volleyball activity classes.

In [1]:
import os
import sys
import cv2
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.ops import roi_align
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_320_fpn, FasterRCNN_MobileNet_V3_Large_320_FPN_Weights
try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

## 1. Configurations
Define category strings and index mapping dictionary lookups.

In [2]:
GROUP_ACTIVITIES = ['r_set', 'r_spike', 'r_pass', 'r_winpoint', 'l_set', 'l_spike', 'l_pass', 'l_winpoint']
PLAYER_ACTIONS = ['waiting', 'setting', 'spiking', 'digging', 'jumping', 'blocking', 'falling', 'moving', 'standing']

group_to_idx = {name: idx for idx, name in enumerate(GROUP_ACTIVITIES)}
action_to_idx = {name: idx for idx, name in enumerate(PLAYER_ACTIONS)}

## 2. Dataset Loader
Re-use the VolleyBallDataset definition with support for loading 9-frame sequence clips.

In [3]:
class VolleyBallDataset(Dataset):
    def __init__(self, split='train', transform=None, data_path='./volleyball/volleyball_/videos', seq_len=None, stride=3):
        self.data_path = data_path
        self.split = split
        self.transform = transform
        self.seq_len = seq_len
        self.stride = stride
        
        self.train_videos = {1, 3, 6, 7, 10, 13, 15, 16, 18, 22, 23, 31, 32, 36, 38, 39, 40, 41, 42, 48, 50, 52, 53, 54}
        self.val_videos = {0, 2, 8, 12, 17, 19, 24, 26, 27, 28, 30, 33, 46, 49, 51}
        self.test_videos = {4, 5, 9, 11, 14, 20, 21, 25, 29, 34, 35, 37, 43, 44, 45, 47}
        
        self.data = []
        self.load_data()

    def load_data(self):
        if not os.path.exists(self.data_path):
            raise FileNotFoundError(f"Data path '{self.data_path}' not found.")
            
        for video_folder in os.listdir(self.data_path):
            if not video_folder.isdigit():
                continue
            video_id = int(video_folder)
            
            if self.split == 'train' and video_id not in self.train_videos:
                continue
            elif self.split == 'val' and video_id not in self.val_videos:
                continue
            elif self.split == 'test' and video_id not in self.test_videos:
                continue
                
            video_path = os.path.join(self.data_path, video_folder)
            if not os.path.isdir(video_path):
                continue
            annotation_file = os.path.join(video_path, 'annotations.txt')
            if not os.path.exists(annotation_file):
                continue

            for row in open(annotation_file, 'r'):
                frame_folder, groupLabel, *playersAnnotations = row.strip().split(' ')
                frame_name = frame_folder.split('.')[0]
                image_frame  = os.path.join(video_path, frame_name, frame_name + '.jpg')
                
                parsed_players = [{
                    'x': int(playersAnnotations[i]),
                    'y': int(playersAnnotations[i+1]),
                    'w': int(playersAnnotations[i+2]),
                    'h': int(playersAnnotations[i+3]),
                    'action': playersAnnotations[i+4]
                } for i in range(0, len(playersAnnotations), 5)]
                
                annotation = {
                    'groupLabel': groupLabel,
                    'playersAnnotations': parsed_players
                }
                self.data.append((image_frame, annotation))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image_frame, annotation = self.data[idx]
        
        if self.seq_len is not None and self.seq_len > 1:
            frame_dir = os.path.dirname(image_frame)
            frame_name = os.path.basename(image_frame).split('.')[0]
            all_frames = sorted([f for f in os.listdir(frame_dir) if f.endswith('.jpg') or f.endswith('.png')])
            
            target_file = os.path.basename(image_frame)
            if target_file in all_frames:
                mid_idx = all_frames.index(target_file)
            else:
                mid_idx = len(all_frames) // 2
                
            half_len = self.seq_len // 2
            offsets = [i * self.stride for i in range(-half_len, half_len + 1)]
            
            images = []
            for offset in offsets:
                sample_idx = mid_idx + offset
                sample_idx = max(0, min(sample_idx, len(all_frames) - 1))
                frame_path = os.path.join(frame_dir, all_frames[sample_idx])
                
                image = cv2.imread(frame_path)
                image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                images.append(image)

            h_orig, w_orig = images[0].shape[:2]
            target_h, target_w = 720, 1280
            scale_x = target_w / w_orig
            scale_y = target_h / h_orig
            
            scaled_players = []
            for player in annotation['playersAnnotations']:
                x = int(player['x'] * scale_x)
                y = int(player['y'] * scale_y)
                w = int(player['w'] * scale_x)
                h = int(player['h'] * scale_y)
                action_str = player['action']
                action_idx = action_to_idx.get(action_str.lower(), 0) if not str(action_str).isdigit() else int(action_str)
                
                scaled_players.append({
                    'x': x, 'y': y, 'w': w, 'h': h,
                    'action': action_str, 'action_idx': action_idx
                })
                
            group_str = annotation['groupLabel']
            group_idx = group_to_idx.get(group_str.lower(), 0) if not str(group_str).isdigit() else int(group_str)
            scaled_annotation = {
                'groupLabel': group_str, 'groupLabel_idx': group_idx, 'playersAnnotations': scaled_players
            }
            
            processed_images = []
            for img in images:
                if h_orig != target_h or w_orig != target_w:
                    img = cv2.resize(img, (target_w, target_h))
                if self.transform:
                    from PIL import Image
                    img = Image.fromarray(img)
                    img = self.transform(img)
                else:
                    img = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1) / 255.0
                processed_images.append(img)
                
            images_tensor = torch.stack(processed_images, dim=0)
            return images_tensor, scaled_annotation
        else:
            image = cv2.imread(image_frame)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            h_orig, w_orig = image.shape[:2]
            target_h, target_w = 720, 1280
            scale_x = target_w / w_orig
            scale_y = target_h / h_orig
            
            scaled_players = []
            for player in annotation['playersAnnotations']:
                x = int(player['x'] * scale_x)
                y = int(player['y'] * scale_y)
                w = int(player['w'] * scale_x)
                h = int(player['h'] * scale_y)
                action_str = player['action']
                action_idx = action_to_idx.get(action_str.lower(), 0) if not str(action_str).isdigit() else int(action_str)
                
                scaled_players.append({
                    'x': x, 'y': y, 'w': w, 'h': h,
                    'action': action_str, 'action_idx': action_idx
                })
                
            group_str = annotation['groupLabel']
            group_idx = group_to_idx.get(group_str.lower(), 0) if not str(group_str).isdigit() else int(group_str)
            scaled_annotation = {
                'groupLabel': group_str, 'groupLabel_idx': group_idx, 'playersAnnotations': scaled_players
            }
            
            if h_orig != target_h or w_orig != target_w:
                image = cv2.resize(image, (target_w, target_h))
            if self.transform:
                from PIL import Image
                image = Image.fromarray(image)
                image = self.transform(image)
            return image, scaled_annotation

## 3. Model Architecture
Define the tracker class and model architecture.

In [ ]:
import os
import torch
import torch.nn as nn
from torchvision import models
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_320_fpn, FasterRCNN_MobileNet_V3_Large_320_FPN_Weights
from torchvision.ops import roi_align

class IoUTracker:
    def __init__(self, use_detector=False, detector_threshold=0.5):
        """
        use_detector: If True, uses Faster R-CNN to track player boxes across frames.
                      If False, propagates the middle frame boxes (constant propagation).
        """
        self.use_detector = use_detector
        self.detector_threshold = detector_threshold
        self.detector = None

    def _get_detector(self, device):
        if self.detector is None and self.use_detector:
            try:
                weights = FasterRCNN_MobileNet_V3_Large_320_FPN_Weights.DEFAULT
                self.detector = fasterrcnn_mobilenet_v3_large_320_fpn(weights=weights).to(device)
                self.detector.eval()
            except Exception as e:
                print(f"Warning: Failed to load pretrained detector: {e}. Falling back to constant box propagation.")
                self.use_detector = False
        return self.detector

    def track(self, images, annotations, device):
        """
        images: (B, seq_len, C, H, W)
        annotations: list of B dicts containing 'playersAnnotations' for the middle frame
        device: torch.device
        
        Returns:
            tracked_boxes_batch: list of length B, where each item is a tensor of shape (seq_len, num_players, 4)
        """
        batch_size, seq_len, C, H, W = images.shape
        mid_idx = seq_len // 2
        tracked_boxes_batch = []
        
        detector = self._get_detector(device)

        # Scale coordinates:
        # Annotations are originally scaled to 1280x720 in dataset __getitem__.
        # We need to scale them to the actual model input image size H, W.
        scale_x = W / 1280.0
        scale_y = H / 720.0

        for i in range(batch_size):
            player_anns = annotations[i].get('playersAnnotations', [])
            num_players = len(player_anns)
            
            # Extract and scale middle frame boxes
            mid_boxes = []
            for player in player_anns:
                x = player['x'] * scale_x
                y = player['y'] * scale_y
                w = player['w'] * scale_x
                h = player['h'] * scale_y
                
                x1 = max(0.0, min(x, W - 1.0))
                y1 = max(0.0, min(y, H - 1.0))
                x2 = max(x1 + 1.0, min(x + w, W))
                y2 = max(y1 + 1.0, min(y + h, H))
                mid_boxes.append([x1, y1, x2, y2])
            
            if len(mid_boxes) == 0:
                # Fallback if no players annotated
                mid_boxes = [[0.0, 0.0, float(W), float(H)]]
                num_players = 1

            mid_boxes_tensor = torch.tensor(mid_boxes, dtype=torch.float32, device=device)
            seq_boxes = torch.zeros((seq_len, num_players, 4), dtype=torch.float32, device=device)
            seq_boxes[mid_idx] = mid_boxes_tensor

            if not self.use_detector or detector is None:
                # Constant box propagation
                for t in range(seq_len):
                    seq_boxes[t] = mid_boxes_tensor
            else:
                # Track forward in time: mid_idx -> mid_idx + 1 -> ...
                for t in range(mid_idx + 1, seq_len):
                    prev_boxes = seq_boxes[t - 1]
                    detections = self._detect_persons(images[i, t], detector, device)
                    if len(detections) == 0:
                        seq_boxes[t] = prev_boxes
                    else:
                        seq_boxes[t] = self._match_boxes(prev_boxes, detections)

                # Track backward in time: mid_idx -> mid_idx - 1 -> ...
                for t in range(mid_idx - 1, -1, -1):
                    next_boxes = seq_boxes[t + 1]
                    detections = self._detect_persons(images[i, t], detector, device)
                    if len(detections) == 0:
                        seq_boxes[t] = next_boxes
                    else:
                        seq_boxes[t] = self._match_boxes(next_boxes, detections)

            tracked_boxes_batch.append(seq_boxes)

        return tracked_boxes_batch

    @torch.no_grad()
    def _detect_persons(self, img, detector, device):
        """
        img: (C, H, W)
        """
        # Unnormalize standard ImageNet scaling back to [0, 1] for the detector
        mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225], device=device).view(3, 1, 1)
        img_unnorm = img * std + mean
        img_unnorm = torch.clamp(img_unnorm, 0.0, 1.0)
        
        outputs = detector([img_unnorm])
        boxes = outputs[0]['boxes']
        labels = outputs[0]['labels']
        scores = outputs[0]['scores']
        
        # Class 1 is 'person' in COCO dataset used by the detector
        mask = (labels == 1) & (scores >= self.detector_threshold)
        return boxes[mask]

    def _match_boxes(self, track_boxes, detections):
        """
        track_boxes: (N, 4)
        detections: (M, 4)
        """
        iou_matrix = self._box_iou(track_boxes, detections)
        N = track_boxes.shape[0]
        matched_boxes = track_boxes.clone()
        
        iou_np = iou_matrix.cpu().numpy()
        matched_detections = set()
        
        for idx in range(N):
            best_det_idx = -1
            best_iou = 0.3  # min IoU threshold for matching
            for j in range(detections.shape[0]):
                if j in matched_detections:
                    continue
                if iou_np[idx, j] > best_iou:
                    best_iou = iou_np[idx, j]
                    best_det_idx = j
            if best_det_idx != -1:
                matched_boxes[idx] = detections[best_det_idx]
                matched_detections.add(best_det_idx)
                
        return matched_boxes

    def _box_iou(self, boxes1, boxes2):
        # area of boxes1
        area1 = (boxes1[:, 2] - boxes1[:, 0]) * (boxes1[:, 3] - boxes1[:, 1])
        # area of boxes2
        area2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])
        
        # intersection
        lt = torch.max(boxes1[:, None, :2], boxes2[:, :2])  # (N, M, 2)
        rb = torch.min(boxes1[:, None, 2:], boxes2[:, 2:])  # (N, M, 2)
        
        wh = (rb - lt).clamp(min=0)  # (N, M, 2)
        inter = wh[:, :, 0] * wh[:, :, 1]  # (N, M)
        
        union = area1[:, None] + area2 - inter
        return inter / torch.clamp(union, min=1e-6)


class GroupActivityRecognitionB4(nn.Module):
    """
    B4: Temporal Model with Person Features
    - Employs a tracker or constant propagation to obtain player boxes across 9 frames.
    - Crops player regions and extracts features with a ResNet-50 backbone.
    - Runs a player-level LSTM to model the temporal sequence of each player's features.
    - Pools features across all players to recognize the group activity.
    """
    def __init__(self, num_group_classes=8, num_action_classes=9, embed_dim=2048, hidden_size=512, num_layers=1, dropout=0.3, pooling='max', crop_size=(112, 112), use_detector=False, freeze_backbone=False, fine_tune_layer4_only=True):
        super(GroupActivityRecognitionB4, self).__init__()
        self.num_group_classes = num_group_classes
        self.num_action_classes = num_action_classes
        self.pooling = pooling.lower()
        self.crop_size = crop_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # Initialize the tracker
        self.tracker = IoUTracker(use_detector=use_detector)
        
        # Load ResNet-50 backbone
        try:
            self.resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        except Exception:
            self.resnet = models.resnet50(pretrained=True)
            
        self.resnet.fc = nn.Identity()
        
        # Freezing/Unfreezing backbone parameters
        if freeze_backbone:
            for param in self.resnet.parameters():
                param.requires_grad = False
        else:
            if fine_tune_layer4_only:
                # Freeze layers 1-3, fine-tune layer 4
                for name, param in self.resnet.named_parameters():
                    if 'layer4' in name:
                        param.requires_grad = True
                    else:
                        param.requires_grad = False
            else:
                # Fine-tune the entire backbone
                for param in self.resnet.parameters():
                    param.requires_grad = True
        
        # LSTM for temporal sequence modeling at player level
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=False
        )
        
        # Classifier for group activities fed by pooled player features
        if dropout > 0:
            self.classifier = nn.Sequential(
                nn.Dropout(p=dropout),
                nn.Linear(hidden_size, num_group_classes)
            )
        else:
            self.classifier = nn.Linear(hidden_size, num_group_classes)

    def forward(self, images, annotations=None):
        """
        images: (B, seq_len, C, H, W)
        annotations: list of B dicts containing player annotations for the middle frame
        """
        batch_size, seq_len, C, H, W = images.shape
        device = images.device

        if annotations is None:
            # Fallback if no annotations provided (e.g. dummy forward)
            dummy_features = torch.zeros(batch_size, self.hidden_size, device=device)
            group_output = self.classifier(dummy_features)
            action_output = torch.zeros(0, self.num_action_classes, device=device)
            return group_output, action_output

        # 1. Track player boxes across frames
        tracked_boxes_batch = self.tracker.track(images, annotations, device)

        # 2. Crop and resize player regions using torchvision.ops.roi_align
        rois = []
        player_counts = []
        
        for i in range(batch_size):
            seq_boxes = tracked_boxes_batch[i]  # (seq_len, num_players, 4)
            num_players = seq_boxes.shape[1]
            player_counts.append(num_players)
            
            for p in range(num_players):
                for t in range(seq_len):
                    x1, y1, x2, y2 = seq_boxes[t, p]
                    rois.append([i * seq_len + t, x1, y1, x2, y2])
                    
        rois_tensor = torch.tensor(rois, dtype=torch.float32, device=device)
        flat_images = images.view(batch_size * seq_len, C, H, W)
        
        crops_tensor = roi_align(
            flat_images,
            rois_tensor,
            output_size=self.crop_size,
            spatial_scale=1.0,
            sampling_ratio=-1,
            aligned=True
        )

        # 3. Extract spatial features using ResNet-50
        # If backbone is completely frozen, run with no_grad to save memory
        is_backbone_training = any(param.requires_grad for param in self.resnet.parameters())
        if is_backbone_training:
            crop_features = self.resnet(crops_tensor)     # (B * num_players * seq_len, 2048)
        else:
            with torch.no_grad():
                crop_features = self.resnet(crops_tensor)
        
        # Reshape to (B * num_players, seq_len, 2048) for LSTM
        total_players = sum(player_counts)
        crop_features = crop_features.view(total_players, seq_len, -1)

        # 4. Process temporal crop sequences per player via LSTM
        lstm_out, (hn, cn) = self.lstm(crop_features)  # (total_players, seq_len, hidden_size)
        player_temporal_features = lstm_out[:, -1, :]  # (total_players, hidden_size)

        # 5. Pool features across individuals for each batch item
        pooled_features_list = []
        player_offset = 0
        for i in range(batch_size):
            num_players = player_counts[i]
            img_player_features = player_temporal_features[player_offset : player_offset + num_players]
            player_offset += num_players
            
            if self.pooling == 'max':
                pooled, _ = torch.max(img_player_features, dim=0)
            elif self.pooling == 'mean':
                pooled = torch.mean(img_player_features, dim=0)
            else:
                pooled_max, _ = torch.max(img_player_features, dim=0)
                pooled_mean = torch.mean(img_player_features, dim=0)
                pooled = pooled_max + pooled_mean
            pooled_features_list.append(pooled)

        pooled_features = torch.stack(pooled_features_list, dim=0)  # (B, hidden_size)

        # 6. Classify group activity
        group_output = self.classifier(pooled_features)
        action_output = torch.zeros(0, self.num_action_classes, device=device)
        
        return group_output, action_output

## 4. Evaluation Utilities
Define evaluation utilities to calculate macro F1-score.

In [5]:
try:
    from sklearn.metrics import f1_score
    def compute_macro_f1(preds, targets):
        return f1_score(targets, preds, average='macro', zero_division=0)
except ImportError:
    def compute_macro_f1(preds, targets):
        classes = np.unique(np.concatenate([preds, targets]))
        if len(classes) == 0: return 0.0
        f1s = []
        for c in classes:
            tp = np.sum((preds == c) & (targets == c))
            fp = np.sum((preds == c) & (targets != c))
            fn = np.sum((preds != c) & (targets == c))
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
            f1s.append(f1)
        return np.mean(f1s) if len(f1s) > 0 else 0.0

## 5. Training and Validation Loops
Single epoch execution helpers.

In [6]:
def train_epoch(model, dataloader, criterion_group, optimizer, device):
    model.train()
    epoch_group_loss = 0.0
    group_correct = 0
    group_total = 0
    all_group_preds = []
    all_group_labels = []
    
    import gc
    loop = tqdm(dataloader, desc="Training", leave=False)
    for images, annotations in loop:
        images = images.to(device)
        batch_group_labels = [ann['groupLabel_idx'] for ann in annotations]
        group_labels = torch.tensor(batch_group_labels, dtype=torch.long, device=device)
        
        optimizer.zero_grad()
        group_outputs, _ = model(images, annotations)
        loss = criterion_group(group_outputs, group_labels)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        _, pred_group = torch.max(group_outputs, 1)
        group_correct += (pred_group == group_labels).sum().item()
        group_total += group_labels.size(0)
        
        all_group_preds.extend(pred_group.cpu().numpy())
        all_group_labels.extend(group_labels.cpu().numpy())
        epoch_group_loss += loss.item() * images.size(0)
        loop.set_postfix(loss=loss.item(), group_acc=group_correct/max(group_total, 1))
        
        # Explicitly clean up memory
        del group_outputs, loss, images, group_labels
        gc.collect()
        torch.cuda.empty_cache()
        
    num_samples = len(dataloader.dataset)
    group_f1 = compute_macro_f1(np.array(all_group_preds), np.array(all_group_labels))
    return {
        'loss': epoch_group_loss / num_samples,
        'acc': (group_correct / group_total) if group_total > 0 else 0.0,
        'f1': group_f1
    }


@torch.no_grad()
def val_epoch(model, dataloader, criterion_group, device):
    model.eval()
    epoch_group_loss = 0.0
    group_correct = 0
    group_total = 0
    all_group_preds = []
    all_group_labels = []
    
    import gc
    for images, annotations in dataloader:
        images = images.to(device)
        batch_group_labels = [ann['groupLabel_idx'] for ann in annotations]
        group_labels = torch.tensor(batch_group_labels, dtype=torch.long, device=device)
        group_outputs, _ = model(images, annotations)
        loss = criterion_group(group_outputs, group_labels)
        
        _, pred_group = torch.max(group_outputs, 1)
        group_correct += (pred_group == group_labels).sum().item()
        group_total += group_labels.size(0)
        
        all_group_preds.extend(pred_group.cpu().numpy())
        all_group_labels.extend(group_labels.cpu().numpy())
        epoch_group_loss += loss.item() * images.size(0)
        
        # Explicitly clean up memory
        del group_outputs, loss, images, group_labels
        gc.collect()
        torch.cuda.empty_cache()
        
    num_samples = len(dataloader.dataset)
    group_f1 = compute_macro_f1(np.array(all_group_preds), np.array(all_group_labels))
    return {
        'loss': epoch_group_loss / num_samples,
        'acc': (group_correct / group_total) if group_total > 0 else 0.0,
        'f1': group_f1
    }

## 6. Runner Function
Main pipeline orchestration wrapper.

In [ ]:
def run_baseline_4():
    epochs = 10
    batch_size = 4
    lr = 1e-4
    data_path = '/kaggle/input/datasets/ahmedmohamed365/volleyball/volleyball_/videos'
    if not os.path.exists(data_path):
        data_path = './volleyball/volleyball_/videos'
        
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Running Baseline 4 on device: {device}")

    # Clean up any leftover GPU memory before starting
    import gc
    gc.collect()
    torch.cuda.empty_cache()

    train_transform = transforms.Compose([
        transforms.Resize((180, 320)),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
        transforms.RandomGrayscale(p=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    val_transform = transforms.Compose([
        transforms.Resize((180, 320)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    def collate_fn(batch):
        images = [item[0] for item in batch]
        annotations = [item[1] for item in batch]
        images = torch.stack(images, 0)
        return images, annotations
    
    train_dataset = VolleyBallDataset(split='train', transform=train_transform, data_path=data_path, seq_len=9, stride=3)
    val_dataset = VolleyBallDataset(split='val', transform=val_transform, data_path=data_path, seq_len=9, stride=3)
    test_dataset = VolleyBallDataset(split='test', transform=val_transform, data_path=data_path, seq_len=9, stride=3)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn, num_workers=0, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn, num_workers=0, pin_memory=True)

    print(f"Train samples: {len(train_dataset)}")
    print(f"Validation samples: {len(val_dataset)}")
    print(f"Test samples: {len(test_dataset)}")

    model = GroupActivityRecognitionB4(
        num_group_classes=len(GROUP_ACTIVITIES),
        num_action_classes=len(PLAYER_ACTIONS),
        embed_dim=2048,
        hidden_size=512,
        num_layers=1,
        dropout=0.3,
        pooling='max',
        use_detector=True,
        freeze_backbone=False,
        fine_tune_layer4_only=True
    )
    model = model.to(device)
    
    backbone_params = []
    head_params = []
    for name, param in model.named_parameters():
        if not param.requires_grad: continue
        if 'classifier' in name or 'lstm' in name:
            head_params.append(param)
        else:
            backbone_params.append(param)
            
    optimizer_groups = []
    if len(backbone_params) > 0:
        optimizer_groups.append({'params': backbone_params, 'lr': lr, 'weight_decay': 1e-4})
    optimizer_groups.append({'params': head_params, 'lr': lr * 10, 'weight_decay': 1e-3})
    
    optimizer = torch.optim.AdamW(optimizer_groups)
    
    criterion_group = nn.CrossEntropyLoss(label_smoothing=0.1)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    os.makedirs('checkpoints', exist_ok=True)
    best_model_path = 'checkpoints/best_model.pth'
    best_val_f1 = 0.0

    history = {
        'train_loss': [], 'train_acc': [], 'train_f1': [],
        'val_loss': [], 'val_acc': [], 'val_f1': []
    }
    
    print("\nStarting Training Loop...")
    for epoch in range(1, epochs + 1):
        print(f"\n--- Epoch {epoch}/{epochs} ---")
        current_lrs = [group['lr'] for group in optimizer.param_groups]
        print(f"Learning Rates - Backbone: {current_lrs[0]:.2e} | Head: {current_lrs[1]:.2e}" if len(current_lrs) > 1 else f"Learning Rates - Head: {current_lrs[0]:.2e}")
        
        train_metrics = train_epoch(model, train_loader, criterion_group, optimizer, device)
        val_metrics = val_epoch(model, val_loader, criterion_group, device)
        
        scheduler.step()
        
        history['train_loss'].append(train_metrics['loss'])
        history['train_acc'].append(train_metrics['acc'])
        history['train_f1'].append(train_metrics['f1'])
        history['val_loss'].append(val_metrics['loss'])
        history['val_acc'].append(val_metrics['acc'])
        history['val_f1'].append(val_metrics['f1'])
        
        print(f"Train Loss: {train_metrics['loss']:.4f} | Train Acc: {train_metrics['acc']*100:.2f}% | Train F1: {train_metrics['f1']:.4f}")
        print(f"Val   Loss: {val_metrics['loss']:.4f} | Val   Acc: {val_metrics['acc']*100:.2f}% | Val   F1: {val_metrics['f1']:.4f}")

        if val_metrics['f1'] > best_val_f1:
            best_val_f1 = val_metrics['f1']
            torch.save(model.state_dict(), best_model_path)
            print(f"Saved new best model to {best_model_path} with Val F1: {best_val_f1:.4f}")
            
    print("\nTraining completed successfully!")


## 7. Execution
Run the training pipeline.

In [ ]:
run_baseline_4()

Running Baseline 4 on device: cuda
Train samples: 2152
Validation samples: 1341
Test samples: 1337
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 199MB/s]



Starting Training Loop...

--- Epoch 1/5 ---
Learning Rates - Head: 1.00e-03


Training:   0%|          | 0/538 [00:00<?, ?it/s]

Downloading: "https://download.pytorch.org/models/fasterrcnn_mobilenet_v3_large_320_fpn-907ea3f9.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_mobilenet_v3_large_320_fpn-907ea3f9.pth



  0%|          | 0.00/74.2M [00:00<?, ?B/s]
  9%|▊         | 6.38M/74.2M [00:00<00:01, 66.8MB/s]
 31%|███       | 23.0M/74.2M [00:00<00:00, 130MB/s] 
 52%|█████▏    | 38.9M/74.2M [00:00<00:00, 147MB/s]
 75%|███████▍  | 55.4M/74.2M [00:00<00:00, 157MB/s]
100%|██████████| 74.2M/74.2M [00:00<00:00, 150MB/s]


Train Loss: 1.4373 | Train Acc: 60.50% | Train F1: 0.1268
Val   Loss: 1.4269 | Val   Acc: 59.88% | Val   F1: 0.1498
Saved new best model to checkpoints/best_model.pth with Val F1: 0.1498

--- Epoch 2/5 ---
Learning Rates - Head: 9.05e-04


Training:   0%|          | 0/538 [00:00<?, ?it/s]

Train Loss: 1.4114 | Train Acc: 60.50% | Train F1: 0.1508
Val   Loss: 1.4118 | Val   Acc: 59.88% | Val   F1: 0.1498

--- Epoch 3/5 ---
Learning Rates - Head: 6.55e-04


Training:   0%|          | 0/538 [00:00<?, ?it/s]